# Notebook 03 — Git Fundamentals

**Module:** 00 — Orientation
**Notebook:** 03 of 13
**Prerequisites:** Notebooks 01–02
**Time estimate:** 90 minutes

Git is the audit trail of this program. Every experiment, every notebook, every paper
note is versioned. This notebook teaches the workflow you will repeat every session.

---
## Step 1 — Motivation

Science has a reproducibility problem. A large fraction of published results cannot
be reproduced because the code that generated them is unavailable, incomplete, or
locked to a specific machine state at a specific moment.

Git is not a silver bullet, but it solves one specific part of this problem:
**the audit trail.** When your work lives in Git:
- You can reconstruct exactly what code produced any result, at any date.
- You can safely experiment (branches) without breaking working code.
- A collaborator, reviewer, or future you can understand what changed and why.

For Track A: every India RA/JRF posting that asks for "GitHub profile" is asking
whether you use Git correctly — not just whether you know the commands.
For Track B: a PhD supervisor reading your repository is reading your commit history
as much as your code.

---
## Step 2 — Intuition

Think of Git as a **time machine for a folder**.

- Every `git commit` takes a snapshot of the folder's state at that moment.
- You can travel back to any snapshot with `git checkout <hash>`.
- Branches are parallel timelines — you can explore an idea on a branch without
  affecting the main timeline, then merge it in (or discard it) when ready.
- The remote (GitHub) is a backup of the timeline that other people can also read.

**The workflow for this program (one cycle = one study session):**
```
1. git pull                        ← sync with remote before starting
2. git checkout -b module-01/nb04  ← create a branch for today's notebook
3. [edit/write notebooks and code]
4. git add <specific files>        ← stage only what belongs together
5. git commit -m "meaningful message"
6. git push -u origin module-01/nb04
7. (when module complete) merge to main, tag
```

---
## Step 3 — Biological Background

*Not applicable to this Git fundamentals notebook.*

**Meta-note:** Version control has a direct biological analogue — DNA is itself a
version-controlled sequence. Mutations are commits; selection is merging; drift is
uncommitted changes that get lost. This is not a rigorous analogy, but it makes the
vocabulary stick.

---
## Step 4 — Mathematical Explanation

*Not applicable to this notebook.*

Git's internal model is a **directed acyclic graph (DAG)** of content-addressed
objects. Each commit hash is the SHA-1 of its content plus its parent hash(es).
This makes the history tamper-evident. You will encounter DAGs formally in
Module 12 (Systems & Network Biology).

---
## Step 5 — Computational Explanation

**Core Git objects (all you need to understand the commands):**

| Object | What it stores |
|--------|---------------|
| `blob` | file content |
| `tree` | directory listing (names + blob hashes) |
| `commit` | tree hash + parent hash + author + message |
| `ref` | a named pointer to a commit (branch, tag, HEAD) |

**The three areas Git cares about:**

```
Working directory ──── git add ───▶ Staging area ──── git commit ───▶ Repository
    (your files)                    (what will go in                   (commit history)
                                     the next commit)
```

`git status` shows which files are in which area.
`git diff` shows what changed in the working directory.
`git diff --staged` shows what changed in the staging area.

---
## Step 6 — Python Implementation

Git is a command-line tool, not a Python library. The cells below use `subprocess`
to run Git commands and capture their output — a useful Python pattern for
scripting shell workflows.

In [ ]:
# Cell 6.1 — Run git commands from Python using subprocess
import subprocess
import sys

def git(args: list[str], cwd: str = ".") -> str:
    """Run a git command and return stdout. Raises on error."""
    result = subprocess.run(
        ["git"] + args,
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{result.stderr}")
    return result.stdout.strip()

# Show current branch and status
print("Current branch:", git(["branch", "--show-current"]))
print()
print("git status:")
print(git(["status", "--short"]))

In [ ]:
# Cell 6.2 — Show recent commit log (last 5 commits)
try:
    log = git(["log", "--oneline", "-5"])
    print("Last 5 commits:")
    print(log if log else "(no commits yet)")
except RuntimeError as e:
    print(f"No commits yet or not in a git repo:\n{e}")

In [ ]:
# Cell 6.3 — Show the remote configuration
try:
    remotes = git(["remote", "-v"])
    print("Remotes:")
    print(remotes if remotes else "(no remotes configured)")
except RuntimeError as e:
    print(f"Remote check failed:\n{e}")

In [ ]:
# Cell 6.4 — Demonstrate the commit message anatomy
# A good commit message for this program has this structure:
#
# <module>/<topic>: <imperative summary> (≤72 chars)
#
# [Optional body: WHY this change was made, not WHAT it does.
#  The diff shows what; the message explains why.]
#
# Examples of GOOD messages:
#   module-00/nb03: add git fundamentals notebook skeleton
#   module-01/nb04: implement gc_content with property-based test
#   module-08: add Smith-Waterman from scratch, benchmarked vs Biopython
#
# Examples of BAD messages:
#   "update"
#   "fixes"
#   "wip"
#   "added stuff"
#
# The discipline of writing a good commit message forces you to think about
# whether the change is atomic and coherent.

good_messages = [
    "module-00/nb03: add git fundamentals notebook skeleton",
    "module-01/nb04: implement gc_content with property-based test",
    "module-08: add Smith-Waterman from scratch, benchmarked vs Biopython",
]
bad_messages = ["update", "fixes", "wip", "added stuff"]

print("Good commit messages:")
for msg in good_messages:
    print(f"  ✓  {msg}")

print("\nBad commit messages (never use these):")
for msg in bad_messages:
    print(f"  ✗  {msg}")

In [ ]:
# Cell 6.5 — Show branch naming convention for this program
branch_examples = [
    ("module-00/orientation",   "All 13 orientation notebooks"),
    ("module-01/numpy",         "NumPy notebooks within Module 01"),
    ("module-01/compbio-utils", "compbio_utils development work"),
    ("module-08/blast-impl",    "BLAST implementation in Module 08"),
    ("portfolio/seq-pipeline",  "Portfolio-level artifact"),
    ("fix/environment-check",   "Bug fix in a script"),
]

print(f"{'Branch name':<35} {'Purpose'}")
print("-" * 65)
for name, purpose in branch_examples:
    print(f"  {name:<33} {purpose}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Draw a text-based Git commit graph
git_graph = """
  Commit history diagram (read bottom to top):

  main ────────────────────────────────────○ (HEAD → main)
                                           │
  module-01/numpy ──────○──────○──────○───╯  (merged)
                         │      │      │
                        nb04   nb05   nb06

  How to read this:
  - Each ○ is a commit (a snapshot)
  - The branch label points to the latest commit on that branch
  - HEAD points to whatever you currently have checked out
  - Merging brings the branch's commits into main
"""
print(git_graph)

---
## Step 8 — Exercises

*Solutions go in `exercises/03_git_exercises.md`.*

**Exercise 1:**
In your terminal, run:
```bash
git log --oneline --graph --all
```
What does the output tell you? If the repository has no commits yet, make the
first commit: add `ENVIRONMENT.md` and `README.md`, write a commit message
using the convention above, and push.

**Exercise 2 (branch practice):**
Create a branch named `module-00/orientation`, make a small edit to any file
in `curriculum/00_orientation/`, commit it with a well-formed message, push the
branch, and then merge it back into `main`. Log each command you ran and
what it did.

**Exercise 3 (commit message practice):**
Write a commit message for the following hypothetical change:
"I added a function called `reverse_complement` to `compbio_utils/sequence.py`
and wrote two pytest tests for it."
Write the message in `exercises/03_git_exercises.md` before looking at
the examples in Cell 6.4.

---
## Quiz — Active Recall

1. What is the difference between `git add .` and `git add <specific file>`?
   When would you prefer each?
2. What does `git diff --staged` show that `git diff` does not?
3. A colleague says "I just used `git push --force` to fix my commit."
   What risk did they take? When is force-push never acceptable?
4. What is the purpose of a branch in the context of this program's workflow?
5. Why does this program avoid `git add .` in general?
   (Hint: think about `.env` files and large datasets.)

---
## Reflection

**Date completed:** ____________________

**Reflection:**

> *[What Git concepts were already familiar? What was new?
> What does a "meaningful commit message" mean to you now vs. before this notebook?
> Did Exercise 2 reveal any gaps in your understanding of branching?]*

---
## References

- [Pro Git book](https://git-scm.com/book/en/v2) — free, authoritative, Chapters 1–3
- [GitHub CLI documentation](https://cli.github.com/manual/)
- `ENVIRONMENT.md` §Git Config — the git configuration commands for this program

---
## Future Work / Open Questions

- Once Module 16 (Research Software Engineering) runs, revisit the branching strategy.
  The current convention is simple; a real CI/CD workflow uses protected branches,
  pull request reviews, and automated test gates.

---
*Next notebook: `03_github_workflow.ipynb`*